In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, gc
import numpy as np, pandas as pd
import torch
from transformers import AutoModel
import librosa

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = BASE + '/audio_1000'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
OUT_DIR = BASE + '/wordlevel_wavlm_features'
CKPT_FILE = BASE + '/wordlevel_checkpoint.json'
os.makedirs(OUT_DIR, exist_ok=True)

done = set()
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        done = set(json.load(f).get('done', []))

audio_map = {}
for f in os.listdir(AUDIO_DIR):
    if not f.endswith(('.m4a', '.wav', '.mp3')):
        continue
    base = f.rsplit('.', 1)[0]
    if ',' in base:
        base = base.split(',')[0]
    audio_map[base] = os.path.join(AUDIO_DIR, f)

label_map = {}
for f in os.listdir(LABEL_DIR):
    if f.endswith('.csv'):
        label_map[f.replace('.csv', '')] = os.path.join(LABEL_DIR, f)

overlap = sorted(set(audio_map.keys()) & set(label_map.keys()) - done)
print(f'Audio: {len(audio_map)} | Labels: {len(label_map)} | To process: {len(overlap)}')
print(f'Already done: {len(done)}')
if len(overlap) == 0:
    print('No videos to process!')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    torch.cuda.empty_cache()
    gc.collect()
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
SR = 16000
print('WavLM ready!')


In [ ]:
def parse_timestamp(ts):
    ts = str(ts).strip().strip('[]')
    p = ts.split(',')
    return float(p[0]), float(p[1])

def extract_one_word(audio_segment, sr=16000):
    if len(audio_segment) < 400:
        return np.zeros(768, dtype=np.float32)
    inputs = wavlm.feature_extractor(audio_segment, sampling_rate=sr, return_tensors='pt')
    input_values = inputs.input_values.to(device)
    with torch.no_grad():
        out = wavlm(input_values).last_hidden_state
        emb = out.mean(dim=1)
    emb = emb.squeeze(0).cpu().numpy()
    del input_values, out
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return emb.astype(np.float32)

print('Extractor ready: one word at a time')


In [ ]:
t0 = time.time()
for i, vid in enumerate(overlap):
    out_file = OUT_DIR + '/' + vid + '_word_features.npy'
    if os.path.exists(out_file):
        print(f'{i+1}/{len(overlap)} {vid}: already exists, skip')
        continue

    df = pd.read_csv(label_map[vid])
    word_times = []
    for _, row in df.iterrows():
        try:
            t0_w, t1_w = parse_timestamp(row['timestamp'])
            word_times.append((t0_w, t1_w))
        except Exception as e:
            continue

    if not word_times:
        print(f'{i+1}/{len(overlap)} {vid}: no valid timestamps, skip')
        continue

    y_full, _ = librosa.load(audio_map[vid], sr=SR, mono=True)
    n_samples = len(y_full)

    all_emb = []
    for j, (t0_w, t1_w) in enumerate(word_times):
        s = int(t0_w * SR)
        e = min(int(t1_w * SR), n_samples)
        if e <= s:
            all_emb.append(np.zeros(768, dtype=np.float32))
            continue
        seg = y_full[s:e]
        emb = extract_one_word(seg, SR)
        all_emb.append(emb)

    feats = np.vstack(all_emb)
    np.save(out_file, feats)
    done.add(vid)

    del y_full, all_emb
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    elapsed = time.time() - t0
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    rem_videos = len(overlap) - i - 1
    rem_min = rem_videos / rate * 60 if rate > 0 else 0
    print(f'{i+1}/{len(overlap)} {vid}: {feats.shape} | done={len(done)} | {rate:.0f}/hr | ETA={rem_min:.0f}min')

    if len(done) % 5 == 0:
        with open(CKPT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(CKPT_FILE, 'w') as f:
    json.dump({'done': list(done)}, f)
print(f'\nDone: {len(done)}/{len(overlap)} videos in {(time.time()-t0)/60:.0f} min')


In [ ]:
feat_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('_word_features.npy')])
print(f'Word-level features: {len(feat_files)} videos')
for f in feat_files[:5]:
    d = np.load(OUT_DIR + '/' + f)
    print(f'  {f}: {d.shape}')
